# عينة 10 ثوانٍ — Why Perfect Railroads Don't Work

قانون العينات: نصنع العينة فقط. إذا نجحت نكمل الباقي.

1. الخلية 1: البيئة (Node + مكتبات Chrome)
2. الخلية 2: المشروع + Blender 5.2
3. الخلية 3: التحقق + تصدير البطاقات
4. الخلية 4: معاينة 6 إطارات (أوقف هنا واعرضها)
5. الخلية 5: الرندر الكامل 300 إطار + التسميات + الحفظ في درايف

بدون صوت (القانون).

In [ ]:
# الخلية 1 — البيئة
import subprocess
v = subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip()
print('node:', v or 'غير موجود')
if not any(v.startswith(x) for x in ('v20', 'v22')):
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
    !apt-get install -y nodejs
!apt-get install -y -qq libnss3 libgbm1 libasound2 libatk-bridge2.0-0 libgtk-3-0 libx11-xcb1 libxcomposite1 libxdamage1 libxrandr2 libcups2 libdrm2 libxkbcommon0 libpango-1.0-0 libcairo2 libegl1 libgl1 2>&1 | tail -1
!node -v && ffmpeg -version | head -1

In [ ]:
# الخلية 2 — المشروع + Blender 5.2
import os
REPO = 'https://github.com/creatanim/tester.git'
!rm -rf /content/tester
!git clone $REPO /content/tester
%cd /content/tester/sample10s
!npm install --no-audit --no-fund 2>&1 | tail -2
# Blender 5.2 (Stage-2 GPU render)
if not os.path.isdir('/content/blender'):
    %cd /content
    !curl -sL -o blender.tar.xz https://download.blender.org/release/Blender5.2/blender-5.2.0-linux-x64.tar.xz
    !tar -xf blender.tar.xz && mv blender-5.2.0-linux-x64 blender && rm blender.tar.xz
!/content/blender/blender -v | head -2
%cd /content/tester/sample10s

In [ ]:
# الخلية 3 — التحقق (40 بوابة) + تصدير بطاقات الشخصيات والكاميرا
%cd /content/tester/sample10s
!python3 scripts/validate_plan.py public/plan.json public/audio/words.json --starter
!node scripts/export-cards.mjs --entry src/scene/cards.ts --story src/scene/story.ts --out out/cards
!ls out/cards | head -8 && echo ... && ls out/cards | wc -l

In [ ]:
# الخلية 4 — معاينة 6 إطارات (EEVEE سريع) — اعرضها ثم تابع
%cd /content/tester/sample10s
!mkdir -p out/blender_smoke out/preview
!/content/blender/blender -b --python-exit-code 1 -P scripts/blender_render.py -- --plan public/plan.json --cards out/cards --out out/blender_smoke --engine EEVEE --samples 6 --scale 1.0 --format JPEG --frames-list 5,45,95,150,225,285 2>&1 | tail -8
import shutil, glob
for f in glob.glob('out/blender_smoke/*.jpg'):
    shutil.copy(f, 'out/preview/')
!cd out/preview && zip -q /content/sample_smoke.zip *.jpg
from google.colab import files
files.download('/content/sample_smoke.zip')
print('معاينة جاهزة — افحص الإطارات الستة')

In [ ]:
# الخلية 5 — الرندر الكامل 300 إطار + التسميات (بدون صوت) + الحفظ
%cd /content/tester/sample10s
!/content/blender/blender -b --python-exit-code 1 -P scripts/blender_render.py -- --plan public/plan.json --cards out/cards --out out/blender_frames --engine EEVEE --samples 10 --scale 1.0 --format JPEG 2>&1 | tail -4
!node scripts/composite_stage2.mjs --frames out/blender_frames --format jpg --words public/audio/words.json --out out/sample10s_1080.mp4
# مطابقة أبعاد الأصل 720x1280
!ffmpeg -y -i out/sample10s_1080.mp4 -vf scale=720:1280:flags=lanczos -c:v libx264 -crf 16 -preset medium -pix_fmt yuv420p -movflags +faststart out/sample10s.mp4 2>&1 | tail -2
!python3 scripts/short_qa.py out/sample10s.mp4 out/qa_report public/plan.json 2>&1 | tail -6
import shutil, os
from google.colab import drive, files
drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/Why Perfect Railroads Remake'
os.makedirs(DEST, exist_ok=True)
shutil.copy('out/sample10s.mp4', DEST + '/sample10s.mp4')
print('saved:', DEST + '/sample10s.mp4')
files.download('out/sample10s.mp4')